# 6. Domainloop stats

Part of the **[Fig. 4 chapter](fig4.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from scipy.stats import zscore
from concurrent.futures import ProcessPoolExecutor, as_completed

import anndata

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
!hicluster domain --cell_table_path coollist_L1_impute25k.tsv --output_prefix L1_impute --resolution 25000 --cpu 20


In [ ]:
adata_raw = anndata.read_h5ad(f'{indir}analysis/domain/L1_raw.boundary.h5ad')
adata_impute = anndata.read_h5ad(f'{indir}analysis/domain/L1_impute.boundary.h5ad')[adata_raw.obs.index]


In [ ]:
tad_stats = []
tad_all_raw = []
tad_all_impute = []
for i,ct in enumerate(adata_raw.obs.index):
    tmp1 = np.repeat(adata_raw.X[i].indices, adata_raw.X[i].data).reshape((-1, 2))
    tmp2 = np.repeat(adata_impute.X[i].indices, adata_impute.X[i].data).reshape((-1, 2))
    tad_stats.append([tmp1.shape[0], (tmp1[:,1]-tmp1[:,0]).mean(), tmp2.shape[0], (tmp2[:,1]-tmp2[:,0]).mean()])
    tmp1 = pd.DataFrame(tmp1)
    tmp1['L1'] = ct
    tmp1['length'] = (tmp1[1] - tmp1[0]) * 25000
    tmp2 = pd.DataFrame(tmp2)
    tmp2['L1'] = ct
    tmp2['length'] = (tmp2[1] - tmp2[0]) * 25000
    tad_all_raw.append(tmp1[['L1', 'length']])
    tad_all_impute.append(tmp2[['L1', 'length']])
    
tad_stats = pd.DataFrame(tad_stats, index=adata_raw.obs.index, 
                         columns=['raw_count', 'raw_length', 'impute_count', 'impute_length'])
tad_stats[['raw_length', 'impute_length']] *= 25000
tad_stats = tad_stats.loc[L1_meta.index]
tad_all_raw = pd.concat(tad_all_raw, axis=0)
tad_all_impute = pd.concat(tad_all_impute, axis=0)


In [ ]:
leg_order = tad_all_raw.groupby('L1')['length'].median().sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(8,3), dpi=300)
# sns.violinplot(loop_all, x='L1', y='length', order=leg_order, inner=None,
#                palette=L1_meta['color'].to_dict(), ax=ax)
sns.boxplot(tad_all_raw, x='L1', y='length', order=leg_order, width=0.6, ax=ax, 
            showfliers=False, palette=L1_meta['color'].to_dict(), 
            # showcaps=False, boxprops={'facecolor': 'None'}, 
            # medianprops={'color': 'w', 'linewidth': 1}
           )
ax.set_xticklabels(leg_order.map(L1_meta['L1_abbr']), rotation=90)
fig.tight_layout()
fig.savefig(f'domain/domain_raw_length_box.pdf', transparent=True)


In [ ]:
leg_order = tad_all_impute.groupby('L1')['length'].median().sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(8,3), dpi=300)
# sns.violinplot(loop_all, x='L1', y='length', order=leg_order, inner=None,
#                palette=L1_meta['color'].to_dict(), ax=ax)
sns.boxplot(tad_all_impute, x='L1', y='length', order=leg_order, width=0.6, ax=ax, 
            showfliers=False, palette=L1_meta['color'].to_dict(), 
            # showcaps=False, boxprops={'facecolor': 'None'}, 
            # medianprops={'color': 'w', 'linewidth': 1}
           )
ax.set_xticklabels(leg_order.map(L1_meta['L1_abbr']), rotation=90)
fig.tight_layout()
fig.savefig(f'domain/domain_impute_length_box.pdf', transparent=True)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4, 3), dpi=300)
# ax.spines['bottom'].set_position('zero')
# ax.spines['right'].set_visible(False)
# ax.spines['top'].set_visible(False)
xticks = np.arange(tad_stats.shape[0])
for i,xx in enumerate(['raw_count', 'impute_count']):
    ax = axes[i]
    leg_order = tad_stats.sort_values(xx).index[::-1]
    tmp = tad_stats.loc[leg_order]
    # bottom = [1e3, 1e3][i]
    bottom = tmp[xx].min()
    height = tmp[xx] - bottom
    bottom -= height.max() * 0.1
    height += height.max() * 0.1
    # if bottom>0:
    bottom = np.ones(tmp.shape[0]) * bottom
    # else:
    #     bottom = np.zeros(tmp.shape[0])
    #     height = tmp[xx]
    ax.bar(x=np.arange(tad_stats.shape[0]), height=height, bottom=bottom, 
           color=tmp.index.map(L1_color), width=0.7)
    ax.set_xticks(xticks)
    ax.set_xticklabels(tmp.index.map(L1_annot), fontsize=6, rotation=90)
    # ax.set_yscale('log')
    ax.set_xlim([-1, tad_stats.shape[0]])
    # ax.set_yscale('log')
    # ax.set_yticks([[1e4, 1e5, 1e6], [1e3, 1e4, 1e5]][i])
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
    ax.set_title(xx.replace('_', ' '), fontsize=7)
    
fig.tight_layout()
fig.savefig(f'domain/domain_length_bar.pdf', transparent=True)


In [ ]:
import cooler
cool = cooler.Cooler(f'{ENTEX_ROOT}/merged_cool_impute/25K/L1/c7-b1.Q.cool')


In [ ]:
cool.info['group_n_cells']

In [ ]:
loop_stats = []
loop_all = []
summit_all = []
for ct in L1_meta.index:
    tmp = pd.read_csv(f'{indir}loop/majortype/{ct}/{ct}.loop.bedpe', sep='\t', header=None, index_col=None)
    tmp2 = pd.read_csv(f'{indir}loop/majortype/{ct}/{ct}.loop_summit.bedpe', sep='\t', header=None, index_col=None)
    loop_stats.append([tmp.shape[0], (tmp[4]-tmp[1]).mean(), tmp2.shape[0], (tmp2[4]-tmp2[1]).mean()])
    tmp['L1'] = ct
    tmp['length'] = tmp[4] - tmp[1]
    tmp2['L1'] = ct
    tmp2['length'] = tmp2[4] - tmp2[1]
    loop_all.append(tmp[['L1', 'length']])
    summit_all.append(tmp2[['L1', 'length']])
        
loop_stats = pd.DataFrame(loop_stats, columns=['pixel_count', 'pixel_length', 'summit_count', 'summit_length'], index=L1_meta.index)
loop_all = pd.concat(loop_all, axis=0)
summit_all = pd.concat(summit_all, axis=0)


In [ ]:
print(loop_all.shape[0], summit_all.shape[0])

In [ ]:
leg_order = loop_all.groupby('L1')['length'].median().sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(8,3), dpi=300)
# sns.violinplot(loop_all, x='L1', y='length', order=leg_order, inner=None,
#                palette=L1_meta['color'].to_dict(), ax=ax)
sns.boxplot(loop_all, x='L1', y='length', order=leg_order, width=0.6, ax=ax, 
            showfliers=False, palette=L1_meta['color'].to_dict(), 
            # showcaps=False, boxprops={'facecolor': 'None'}, 
            # medianprops={'color': 'w', 'linewidth': 1}
           )
ax.set_xticklabels(leg_order.map(L1_meta['L1_abbr']), rotation=90)
fig.tight_layout()
fig.savefig(f'diff_loop/loop_length_box.pdf', transparent=True)


In [ ]:
leg_order = summit_all.groupby('L1')['length'].median().sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(8,3), dpi=300)
# sns.violinplot(loop_all, x='L1', y='length', order=leg_order, inner=None,
#                palette=L1_meta['color'].to_dict(), ax=ax)
sns.boxplot(summit_all, x='L1', y='length', order=leg_order, width=0.6, ax=ax, 
            showfliers=False, palette=L1_meta['color'].to_dict(), 
            # showcaps=False, boxprops={'facecolor': 'None'}, 
            # medianprops={'color': 'w', 'linewidth': 1}
           )
ax.set_xticklabels(leg_order.map(L1_meta['L1_abbr']), rotation=90)
fig.tight_layout()
fig.savefig(f'diff_loop/loop_summit_length_box.pdf', transparent=True)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4, 3), dpi=300)
# ax.spines['bottom'].set_position('zero')
# ax.spines['right'].set_visible(False)
# ax.spines['top'].set_visible(False)
xticks = np.arange(loop_stats.shape[0])
for i,xx in enumerate(['pixel_count', 'summit_count']):
    ax = axes[i]
    leg_order = loop_stats.sort_values(xx).index[::-1]
    tmp = loop_stats.loc[leg_order]
    bottom = [1e4, 1e3][i]
    # bottom = tmp[xx].min()
    height = tmp[xx] - bottom
    # bottom -= height.max() * 0.1
    # height += height.max() * 0.1
    # if bottom>0:
    bottom = np.ones(tmp.shape[0]) * bottom
    # else:
    #     bottom = np.zeros(tmp.shape[0])
    #     height = tmp[xx]
    ax.bar(x=np.arange(loop_stats.shape[0]), height=height, bottom=bottom, 
           color=tmp.index.map(L1_color), width=0.7)
    ax.set_xticks(xticks)
    ax.set_xticklabels(tmp.index.map(L1_annot), fontsize=6, rotation=90)
    # ax.set_yscale('log')
    ax.set_xlim([-0.5, loop_stats.shape[0]-0.5])
    ax.set_yscale('log')
    ax.set_yticks([[1e4, 1e5, 1e6], [1e3, 1e4, 1e5]][i])
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=6)
    ax.set_title(xx.replace('_', ' '), fontsize=7)
    
fig.tight_layout()
fig.savefig(f'diff_loop/loop_stats_bar.pdf', transparent=True)


In [ ]:
from schicluster.loop.loop_calling import find_summit
bedpe_cols = ['chrom', 'x1', 'x2', 'chrom2', 'y1', 'y2', 'E']
loop = pd.read_csv(f'{indir}loop/majortype/c1/c1.loop.bedpe', header=None, index_col=None, sep='\t')
loop.columns = bedpe_cols.copy()


In [ ]:
dist_thres, resolution = 20000, 10000
summit = pd.concat([
            find_summit(
                loop=sub_df, res=resolution, dist_thres=dist_thres // resolution)
            for chrom, sub_df in loop.groupby('chrom')
        ],
            axis=0)

In [ ]:
(summit['size']>1).sum()

In [ ]:
looppath = pd.read_csv(f'{indir}loop/subtype/subtype-loop-path.csv', header=0, index_col=0)


In [ ]:
looppath

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

def download(group, ct, path):
    cmd = f'gsutil cp {path}/{group}/{group}.loop.bedpe {indir}loop/subtype/{ct}.loop.bedpe'
    os.system(cmd)
    return ct
    
def check(group, ct, path):
    cmd = f'gsutil ls {path}/Success > success_{ct}.txt'
    os.system(cmd)
    return ct
    
cpu = 30
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for group,ct,path in looppath.values:
        future = executor.submit(
            check,
            group=group,
            ct=ct,
            path=path,
        )
        futures[future] = ct

    result = {}
    for future in as_completed(futures):
        ct = futures[future]
        result[ct] = future.result()
        print(f'{ct} finished')
    

In [ ]:
for f in $(cat success*txt | awk -F'/' '{print $(NF-1)}' | cut -d- -f1-2 | sort); do wc -l ${f}.loop.bedpe; done
